In [ ]:
import os
from pathlib import Path

# Change working directory to parent directory
import sys
# Locate the repo root (folder containing the `core` package) and make it importable
_root = Path.cwd()
if not (_root / "core").is_dir():
    _root = _root.parent
os.chdir(_root)                       # data/ and outputs/ paths resolve from the repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))    # so `core`, `classifiers`, `app`, ... import in any kernel

# Optional: verify
print(Path.cwd())

# 1. Imports

In [ ]:
# --- vep path bootstrap (auto-added; safe to re-run) ---
import os, sys
from pathlib import Path
def _vep_root():
    cands = []
    _p = globals().get('__vsc_ipynb_file__')           # VSCode exposes the notebook path
    if _p: cands.append(Path(_p).resolve().parent)
    cands.append(Path.cwd().resolve())
    for s in cands:
        for c in [s, *s.parents]:                       # search upward for the repo root
            if (c / 'core').is_dir() and (c / 'classifiers').is_dir():
                return c
    return Path.cwd().resolve()
_root = _vep_root()
os.chdir(_root)                                          # data/ and outputs/ resolve from repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))                      # makes core/classifiers/app importable
# --- end vep path bootstrap ---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.decomposition import PCA, FastICA
from sklearn.linear_model import LogisticRegression

# --- Custom imports ---
from core.helpers import load_dataset_paths, compute_average_signal, load_preprocessed_signal, get_data, load_blind_signal
from classifiers.RandomForest_classifier import RandomForest
from core.preprocessing import preprocess_signal, preprocess_save_all
from core import config

# 2. Loading Files

In [ ]:
DEVICES = config.DEVICES
LABELS = config.LABELS
TMAX = config.TMAX
DELAY = 0

preprocess_save_all(
    normalize=True,
    tmax=TMAX,
    SNR_filtering=True,
    do_artifact_removal=False,
    include_blind=True,
)

all_paths_preprocessed = load_dataset_paths()
blind_files_sorted = load_blind_signal()

for device in DEVICES:
    print(f"\n{device}:")
    for label in LABELS:
        print(f"{label}: {len(all_paths_preprocessed[device][label])}")

print(f"\nBLIND: {len(blind_files_sorted)}")

# 3. Classification

## 3.1. Train Prima Test Prima

In [ ]:
X, labels, raw_X, file_list = get_data(device="PRIMA_LE_DA", classes=2)

rf = RandomForest()
y_true, y_pred = rf.fit_cv(X, labels, n_splits=10, use_pca=False, use_ica=False)
results = rf.evaluate(y_true, y_pred)

#print results
print("Results:")
print(f"Balanced Accuracy: {results['balanced_accuracy']:.4f}")
print(f"F1 Score: {results['f1']:.4f}")

## 3.2. Train Prima Test MP20

In [ ]:
X_train, y_train, _, _ = get_data(device="PRIMA_LE_DA",  classes=2)
X_test, y_test, _, _ = get_data(device="MP20_LE_DA",  classes=2)


rf = RandomForest()
y_true, y_pred, _, _ = rf.fit_train_val(X_train, y_train, X_test, y_test, use_pca=False, use_ica=False)
results = rf.evaluate(y_true, y_pred)

#print results
print("Results:")
print(f"Balanced Accuracy: {results['balanced_accuracy']:.4f}")
print(f"F1 Score: {results['f1']:.4f}")